# Групповой анализ PD-эксперимента

Ноутбук считает метрики айтрекинга и баллы опросников по всем участникам.

**Условия:** flat vs sections (контрбалансировка по `disclosure_order`)  
**Датасеты:** alpha (A1–A6) vs beta (B1–B6) (контрбалансировка по `dataset_order`)  
**Ключевые ET-метрики:** число фиксаций, длит. фиксации, длина сканпути, диаметр зрачка

**Три подвыборки:**
- `df_all` (N=10) — все участники, референс
- `df_good` (N≤10) — участники с допустимым % невалидных данных (для ET-анализа)
- `df_balanced` (N=8) — сбалансированная 2×2 выборка (для корреляций ET×NASA-TLX)

Запускать из корня проекта или из `analysis/`.

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

# openpyxl нужен для pd.ExcelWriter
import importlib
if importlib.util.find_spec('openpyxl') is None:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'], check=True)

import os, sys, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats
from IPython.display import display

_cwd = os.getcwd()
if os.path.basename(_cwd) == 'analysis':
    PROJECT_ROOT   = os.path.dirname(_cwd)
    _ANALYSIS_DIR  = _cwd
else:
    PROJECT_ROOT   = _cwd
    _ANALYSIS_DIR  = os.path.join(_cwd, 'analysis')

for _p in [PROJECT_ROOT, _ANALYSIS_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pd_analysis import (
    DATA_ROOT, TASK_ORDER, KEY_METRICS, ROLL_WIN,
    NASA_TLX_CSV, QUESTIONNAIRES_CSV, CONDITIONS_CSV,
    load_all_participants, load_conditions, load_tasks_meta,
    demographic_stats,
    summary_by_condition, summary_by_task,
    score_stai, score_nasa_tlx,
    compute_data_quality_all, build_balanced_subsample, wilcoxon_condition_test,
    lmm_condition_test,
    spearman_nasa_eyetrack, spearman_by_split, plot_spearman_heatmap,
    apply_fdr_to_spearman,
    _participants, _pupil_series, _pupil_series_split,
    plot_pupil_time_participant, plot_pupil_by_task, plot_pupil_by_group, plot_pupil_traces,
    _ET_METRICS, _NASA_CORR_COLS,
)
from analysis_pd import extract_task_gaze
from analysis import load_gaze_data

print('Участники:', _participants())
print(f'NASA TLX CSV: {NASA_TLX_CSV}  (существует: {os.path.isfile(NASA_TLX_CSV)})')
print(f'Conditions CSV: {CONDITIONS_CSV}  (существует: {os.path.isfile(CONDITIONS_CSV)})')

---
## Секция 1. Загрузка данных айтрекинга

**Структура данных**: 10 участников × 2 блока × 6 задач = 120 наблюдений на уровне задания.  
Каждая строка — один trial (task): метрики агрегированы по всему времени выполнения задачи.  
Условие (`condition`: flat/sections) и датасет (`dataset`: alpha/beta) определяются из `experiment_conditions.csv`.

**Ключевые колонки**:
- `fixation_count`, `fixation_duration_mean`, `scanpath_length` — метрики зрительного внимания
- `pupil_mean_avg`, `pupil_pct_change_avg`, `baseline_avg_mean` — метрики зрачка (когнитивная нагрузка)
- `task_type` — тип задачи: Lookup / Comparison / Diagnosis
- `block_num` — номер блока (1 или 2); используется как ковариата для контроля порядкового эффекта


In [ ]:
df_all = load_all_participants()
print(f'Строк: {len(df_all)}, участников: {df_all["participant"].nunique()}')
df_all.head(3)

---
## Секция 1.5. Препроцессинг зрачка

Реализует пайплайн по **Mathôt & Vilotijević (2023)** и **Kret & Sjak-Shie (2019)**:

1. **Бинокулярное усреднение** (at-least-one-eye-valid) — хотя бы один глаз трекается → используем его; оба → среднее.
2. **Speed filter** |ΔD| > 0.5 ед/сэмпл → NaN, одинаково для ITI-baseline и task. Порог p90 по всем участникам — распределение унимодально, порог выбран консервативно.
3. **Margin ±2 сэмпла** вокруг каждого NaN — убирает краевые blink-артефакты.
4. **Кубическая интерполяция** гэпов ≤ 250 мс (≤ 15 сэмплов при 60 Гц); длинные гэпы — NaN.
5. **Outlier pass ±3 SD** — финальное удаление глобальных выбросов.

**Три варианта baseline-коррекции** (primary — субтрактивный, Mathôt et al. 2018):
- `pupil_subtractive_avg` = mean_task − mean_baseline ← **primary**
- `pupil_z_avg` = (mean_task − mean_baseline) / SD_baseline ← sensitivity
- `pupil_pct_change_avg` = (mean_task − mean_baseline) / mean_baseline × 100 ← legacy/divisive

**QC-поля:** `task_pct_unrecoverable`, `baseline_pct_unrecoverable` — % сэмплов, которые не удалось восстановить.

In [ ]:
from pupil_preprocessing import calibrate_from_data, DEFAULT_PARAMS
import os

_speed_plot = os.path.join(DATA_ROOT, 'pupil_speed_distribution.png')
_current_thr = DEFAULT_PARAMS['speed_thr_px_per_sample']
print(f"Speed-filter threshold из DEFAULT_PARAMS: {_current_thr}")
print("Запускаем calibrate_from_data() — анализ распределения |ΔD| по всем TSV...")

cal = calibrate_from_data(
    DATA_ROOT,
    save_plot=_speed_plot,
    verbose=True,
    current_thr=_current_thr,
)
_recommended_thr = cal['recommended_thr']
print(f"\nВыбранный порог: {_current_thr} (p90 AVG, консервативный)")
print(f"Авто-рекомендация: {_recommended_thr:.3f} (не используется — распределение унимодально, "
      "авто-порог физиологически не обоснован)")

from IPython.display import Image
display(Image(_speed_plot))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# baseline_avg_mean из df_all — уже посчитан новым пайплайном
bl_col = 'baseline_avg_mean'
if bl_col not in df_all.columns:
    print(f"Колонка {bl_col!r} не найдена в df_all — пропускаем")
else:
    pids_sorted = sorted(df_all['participant'].unique())
    fig, axes = plt.subplots(1, len(pids_sorted), figsize=(18, 4), sharey=True)
    fig.suptitle('Baseline зрачка (ITI) per participant — диагностика выбросов трайлов', fontsize=12)

    for ax, pid in zip(axes, pids_sorted):
        vals = df_all[df_all['participant'] == pid][bl_col].dropna().values
        if len(vals) == 0:
            ax.set_title(pid, fontsize=8)
            ax.text(0.5, 0.5, 'нет данных', ha='center', va='center', transform=ax.transAxes, fontsize=7)
            continue
        mu, sd = vals.mean(), vals.std(ddof=1) if len(vals) > 1 else 0
        ax.scatter(range(len(vals)), vals, s=20, color='#2266cc', zorder=3)
        ax.axhline(mu,       color='black', lw=1,   ls='-',  label=f'M={mu:.2f}')
        ax.axhline(mu + 2*sd, color='red',   lw=0.8, ls='--', label='+2SD')
        ax.axhline(mu - 2*sd, color='red',   lw=0.8, ls='--', label='−2SD')
        ax.set_title(pid, fontsize=8)
        ax.set_xlabel('Трайл', fontsize=7)
        ax.tick_params(labelsize=6)
        # Highlight outlier trials
        for j, v in enumerate(vals):
            if abs(v - mu) > 2 * sd:
                ax.scatter(j, v, s=40, color='red', zorder=4)

    axes[0].set_ylabel('baseline_avg_mean (мм)', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_ROOT, 'pd_baseline_per_participant.png'), dpi=120, bbox_inches='tight')
    plt.show()
    print("Красные точки = трайлы за пределами ±2 SD внутри участника → кандидаты на exclusion")

In [ ]:
from pupil_preprocessing import clean_gp3_trace
from analysis_pd import extract_task_gaze, extract_iti_gaze, load_gaze_data as _lgd
from analysis import load_gaze_data
import matplotlib.pyplot as plt
import numpy as np

DEMO_PID = '2899'  # участник с хорошим качеством трекинга

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle(f'Участник {DEMO_PID} — очищенный сигнал зрачка (новый пайплайн, синий) '
             'vs сырой бинокулярный (серый)', fontsize=11)

TASK_ORDER_LOCAL = ['A1','A2','A3','A4','A5','A6','B1','B2','B3','B4','B5','B6']
pid_dir = os.path.join(DATA_ROOT, DEMO_PID)

gaze_cache = {}
for bn in (1, 2):
    gp = os.path.join(pid_dir, f'gaze_{DEMO_PID}_pd_b{bn}_trial.tsv')
    if os.path.isfile(gp):
        gaze_cache[bn] = load_gaze_data(gp)

for ax_idx, task_id in enumerate(TASK_ORDER_LOCAL):
    ax = axes.flatten()[ax_idx]
    bn = 1 if task_id.startswith('A') else 2
    bdf = gaze_cache.get(bn)
    if bdf is None:
        ax.set_visible(False)
        continue

    task_df = extract_task_gaze(bdf, task_id)
    if len(task_df) == 0:
        ax.set_title(f'{task_id} (нет данных)', fontsize=8)
        continue

    # Сырой сигнал (both-eyes-valid, без очистки)
    t0 = task_df['TIME'].iloc[0]
    t  = task_df['TIME'].values - t0
    both = (task_df['LPV'] == 1) & (task_df['RPV'] == 1)
    raw = np.full(len(task_df), np.nan)
    raw[both] = (task_df.loc[both, 'LPD'].values + task_df.loc[both, 'RPD'].values) / 2
    ax.plot(t, raw, color='#aaaaaa', lw=0.6, alpha=0.5, label='сырой')

    # Очищенный сигнал
    ct = clean_gp3_trace(task_df, drop_first_sec=0)
    clean = ct.pupil_clean
    valid_mask = ~np.isnan(clean)
    if valid_mask.any():
        ax.plot(t[valid_mask], clean[valid_mask], color='#2266cc', lw=1.0, alpha=0.85, label='очищен')

    pct_ok = ct.pct_valid_final
    ax.set_title(f'{task_id}  ({pct_ok:.0f}% valid)', fontsize=8)
    ax.set_xlabel('Время (с)', fontsize=7)
    ax.tick_params(labelsize=6)

axes[0][0].legend(fontsize=6, loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, f'pd_pupil_clean_traces_{DEMO_PID}.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats
from pd_analysis import compute_data_quality_all, QUALITY_THRESHOLD

# df_good формируется в Секции 2; здесь вычисляем временно для предварительного QC-анализа
_qdf_tmp = compute_data_quality_all(df_all, threshold=QUALITY_THRESHOLD)
_good_pids_tmp = _qdf_tmp[~_qdf_tmp['excluded_pupil']]['participant'].tolist()
_df_good_tmp = df_all[df_all['participant'].isin(_good_pids_tmp)].copy()

qc_col = 'task_pct_unrecoverable'
if qc_col not in _df_good_tmp.columns:
    print(f"{qc_col!r} не найдена в df_good — пропускаем")
else:
    print(f"=== Wilcoxon: {qc_col} ~ condition (flat vs sections) ===")
    print("Проверяем, не конфаундировано ли качество трекинга условием.\n")

    agg = (_df_good_tmp.groupby(['participant', 'condition'])[qc_col]
                       .mean().reset_index())
    flat_s = agg[agg['condition'] == 'flat'].set_index('participant')[qc_col]
    sect_s = agg[agg['condition'] == 'sections'].set_index('participant')[qc_col]
    common = flat_s.index.intersection(sect_s.index)

    fv = flat_s.loc[common].values
    sv = sect_s.loc[common].values
    print(f"  flat:     M={fv.mean():.2f}%  SD={fv.std():.2f}%")
    print(f"  sections: M={sv.mean():.2f}%  SD={sv.std():.2f}%")
    print(f"  N пар: {len(common)}")

    if len(common) >= 4:
        W, p = stats.wilcoxon(fv, sv, alternative='two-sided')
        sig = "⚠ ЗНАЧИМО — возможен confound!" if p < 0.05 else "✓ незначимо (p ≥ 0.05)"
        print(f"  Wilcoxon W={W:.1f}, p={p:.4f}  →  {sig}")
        if p < 0.05:
            print("  Вывод: качество трекинга зависит от условия → необходима осторожность")
            print("         при интерпретации зрачковых различий flat vs sections.")
    else:
        print("  Недостаточно пар для теста.")

In [ ]:
from pd_analysis import wilcoxon_condition_test, compute_data_quality_all, QUALITY_THRESHOLD

# df_good формируется в Секции 2; здесь вычисляем временно
_qdf_tmp = compute_data_quality_all(df_all, threshold=QUALITY_THRESHOLD)
_good_pids_tmp = _qdf_tmp[~_qdf_tmp['excluded_pupil']]['participant'].tolist()
_df_good_tmp = df_all[df_all['participant'].isin(_good_pids_tmp)].copy()

print("=== Групповой анализ зрачка (df_good, новый пайплайн) ===\n")

PUPIL_3 = [m for m in ['pupil_pct_change_avg', 'pupil_z_avg', 'pupil_subtractive_avg']
           if m in _df_good_tmp.columns]

print("Средние по условию:")
cond_agg = (_df_good_tmp.groupby(['participant', 'condition'])[PUPIL_3].mean()
                        .groupby('condition').agg(['mean', 'std']))
cond_agg.columns = ['_'.join(c) for c in cond_agg.columns]
display(cond_agg.round(4))

print("\nWilcoxon flat vs sections (H1: sections > flat, alternative='less'):")
wres = wilcoxon_condition_test(_df_good_tmp, metrics=PUPIL_3,
                               alternative='less', n_boot=2000, n_perm=2000, seed=42)
display(wres[['metric', 'M_flat', 'M_sections', 'N_pairs', 'W',
              'p_raw', 'p_perm', 'rank_biserial_r', 'cohens_d',
              'p_fdr', 'significant_fdr']].round(4))

print("\nВывод H1 (зрачок в sections > flat):")
for _, row in wres.iterrows():
    direction = "sections > flat" if row['M_sections'] > row['M_flat'] else "sections ≤ flat"
    note = "✓ подтверждена" if row['significant_fdr'] else "✗ не подтверждена"
    print(f"  {row['metric']:<30}: {direction}, p={row['p_raw']:.4f}, d={row['cohens_d']:.3f} → H1 {note}")

---
## Секция 2. Качество данных айтрекинга и формирование подвыборок

**Невалидный сэмпл зрачка** = LPV=0 AND RPV=0 (оба глаза не трекаются).  
**Невалидный сэмпл гейза** = FPOGV=0 (позиция взгляда не определена).  

Участники с `pct_invalid_pupil > QUALITY_THRESHOLD * 100` попадают в `excluded_pupil=True`.

In [ ]:
# Порог исключения (легко изменить)
QUALITY_THRESHOLD = 0.30

print('Вычисляем качество данных (может занять ~1 мин)...')
quality_df = compute_data_quality_all(df_all, threshold=QUALITY_THRESHOLD)
display(
    quality_df.style
    .format({'pct_invalid_pupil': '{:.1f}%', 'pct_invalid_gaze': '{:.1f}%'})
    .apply(lambda _: [
        'background-color: #ffcccc' if quality_df['excluded_pupil'].iloc[i] else ''
        for i in range(len(quality_df))
    ], axis=0)
)
print(f'\nПорог: {QUALITY_THRESHOLD*100:.0f}%  |  '
      f'Исключено: {quality_df["excluded_pupil"].sum()}  |  '
      f'Остаток (good): {(~quality_df["excluded_pupil"]).sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pids   = quality_df['participant'].tolist()
pupil_pct = quality_df['pct_invalid_pupil'].fillna(0).tolist()
gaze_pct  = quality_df['pct_invalid_gaze'].fillna(0).tolist()

y = range(len(pids))
ax.barh(y, pupil_pct, color=[
    '#cc4444' if q else '#4488cc'
    for q in quality_df['excluded_pupil']
], alpha=0.8, label='% невалидных записей зрачка')
ax.axvline(QUALITY_THRESHOLD * 100, color='black', ls='--', lw=1.5,
           label=f'Порог {QUALITY_THRESHOLD*100:.0f}%')
ax.set_yticks(list(y))
ax.set_yticklabels(pids)
ax.set_xlabel('% невалидных сэмплов зрачка')
ax.set_title('Качество записи по участникам\n(красный = excluded от df_good)')
ax.legend(fontsize=9)
plt.tight_layout()
display(fig)
plt.close(fig)


In [ ]:
bad_pids      = quality_df[quality_df['excluded_pupil']]['participant'].tolist()
good_pids     = quality_df[~quality_df['excluded_pupil']]['participant'].tolist()
df_good       = df_all[df_all['participant'].isin(good_pids)].copy()

# df_balanced: по 2 участника из каждой ячейки 2×2 дизайна
# (выбираем с наименьшим % невалидных данных из ячеек N=3)
balanced_pids = build_balanced_subsample(quality_df)
df_balanced   = df_all[df_all['participant'].isin(balanced_pids)].copy()

print('Подвыборки:')
print(f'  df_all:      N={df_all["participant"].nunique()} участников')
print(f'  df_good:     N={df_good["participant"].nunique()} участников  {good_pids}')
print(f'  df_balanced: N={df_balanced["participant"].nunique()} участников  {balanced_pids}')
print(f'\nИсключены из df_good: {bad_pids}')

---
## Секция 3. Описательная статистика — Демография

**Характеристика выборки**: N=10, возраст M=27.3 (SD=4.0, диапазон 21–35), 70% мужчин.  
Все участники — IT-специалисты с опытом работы с инфраструктурой.

**Следствия для интерпретации**:
- Выборка **гомогенна**: экспертиза снижает внешнюю валидность — результаты не обобщаются напрямую на новичков или нетехнические аудитории.
- Низкое базовое качество сна (M=4.3/7) может увеличивать дисперсию зрачкового ответа — источник нестабильности в pupil-метриках.
- Отсутствие наивных пользователей позволяет изолировать эффект представления информации (flat vs sections) от эффекта обучения задаче.

Тревожность (STAI) проверяется в секции 4a: при высоком значении тревога может быть конфаундером зрачкового расширения.


In [ ]:
demo = demographic_stats()

print(f"N = {demo['n']}")
print()
age = demo['age']
print(f"Возраст: M={age['mean']}, SD={age['sd']}, диапазон {age['min']}–{age['max']}"
      + (f", пропуски={age['n_missing']}" if age['n_missing'] > 0 else ''))
print(f"Пол: {demo['sex']}")
print(f"Профессия: {demo['profession']}")
print()
it  = demo['it_exp']
inf = demo['infra_exp']
print(f"IT-опыт (лет):    M={it['mean']}, SD={it['sd']}, {it['min']}–{it['max']}")
print(f"Инфраструктурный опыт (лет): M={inf['mean']}, SD={inf['sd']}, {inf['min']}–{inf['max']}")
print(f"Уровень знакомства с инфраструктурой:    {demo['infra_level']}")
slp = demo['sleep_quality']
print(f"Качество сна:     M={slp['mean']}, SD={slp['sd']}")
print()
print("Контактные линзы/очки (важно для качества ET):")
lens_df = pd.DataFrame.from_dict(demo['in_lenses'], orient='index', columns=['Линзы/очки'])
lens_df.index.name = 'ID'
lens_df = lens_df.join(
    quality_df.set_index('participant')[['pct_invalid_pupil', 'excluded_pupil']]
)
display(lens_df)

---
## Секция 4. Описательная статистика — Опросники

### 4a. STAI (ситуативная тревожность, адаптация Ханина)
20 пунктов, суммарный балл 20–80. Норма: <31 = очень низкая, 31–45 = умеренная, >45 = высокая.

In [ ]:
stai_df = score_stai()
display(stai_df.round(1))

valid = stai_df['stai_score'].dropna()
print(f'\nSTAI: M={valid.mean():.1f}, SD={valid.std():.1f}, '
      f'диапазон {valid.min():.0f}–{valid.max():.0f}')
print('Уровни:', stai_df['stai_level'].value_counts().to_dict())

### 4b. NASA-TLX

5 шкал: mental, temporal, performance, effort, frustration.  
Невзвешенная = среднее 5 шкал; взвешенная = Σ(score × weight) / 10.

**Две выборки** (требуется секция 2 — `balanced_pids`):
- **Полная (`nasa_df`, N=10)** — все участники, 20 строк (2 блока × участник); primary для H3 и общего обзора.
- **Сбалансированная (`nasa_balanced`, N=8)** — 2 участника из каждой ячейки 2×2 (`disclosure_order` × `dataset_order`); согласована с `df_balanced` для корреляций ET×NASA-TLX.

In [ ]:
nasa_df = score_nasa_tlx()
nasa_balanced = nasa_df[nasa_df['id'].isin(balanced_pids)]

display(nasa_df[['id','block','condition','dataset',
                 'nasa_unweighted','nasa_weighted',
                 'score_mental','score_temporal','score_performance',
                 'score_effort','score_frustration']].round(2))

print(f"NASA-TLX невзвешенный: M={nasa_df['nasa_unweighted'].mean():.1f}, "
      f"SD={nasa_df['nasa_unweighted'].std():.1f}")
print(f"NASA-TLX взвешенный:   M={nasa_df['nasa_weighted'].mean():.1f}, "
      f"SD={nasa_df['nasa_weighted'].std():.1f}")
print(f'\nПо условию (nasa_balanced, N={nasa_balanced["id"].nunique()}):')
display(nasa_balanced.groupby('condition')[['nasa_unweighted','nasa_weighted']]
        .agg(['mean','std']).round(2))
print(f'\nПо датасету (nasa_balanced, N={nasa_balanced["id"].nunique()}):')
display(nasa_balanced.groupby('dataset')[['nasa_unweighted','nasa_weighted']]
        .agg(['mean','std']).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('NASA-TLX по условию и датасету (nasa_balanced, N=8)', fontsize=12)

for ax, grp_col, colors in [
    (axes[0], 'condition', ['#2266cc', '#cc4400']),
    (axes[1], 'dataset',   ['#55aa44', '#9944cc']),
]:
    grp = nasa_balanced.groupby(grp_col)[['nasa_unweighted', 'nasa_weighted']].mean()
    grp_std = nasa_balanced.groupby(grp_col)[['nasa_unweighted', 'nasa_weighted']].std()
    x = np.arange(len(grp))
    w = 0.35
    ax.bar(x - w/2, grp['nasa_unweighted'], w, yerr=grp_std['nasa_unweighted'],
           color=colors, alpha=0.75, capsize=5, label='Невзвеш.')
    ax.bar(x + w/2, grp['nasa_weighted'],   w, yerr=grp_std['nasa_weighted'],
           color=colors, alpha=0.45, capsize=5, hatch='///', label='Взвеш.')
    ax.set_xticks(x)
    ax.set_xticklabels(grp.index)
    ax.set_title(grp_col.capitalize())
    ax.set_ylabel('NASA-TLX балл')

plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_nasa_tlx.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)

---
## Секция 5. Контрбалансировка — проверка дизайна

Дизайн 2×2: `disclosure_order` (порядок условий) × `dataset_order` (порядок датасетов).  
При N=10 ячейки могут быть неравными → датасет и условие **не ортогональны** в данной выборке.  
Это важно при интерпретации корреляций ET×NASA-TLX по срезам.

In [ ]:
cond_full = pd.read_csv(CONDITIONS_CSV)
cond_full['id'] = cond_full['id'].astype(str)

# Таблица ячеек 2×2
cell_table = cond_full.groupby(['disclosure_order', 'dataset_order']).agg(
    N=('id', 'count'),
    IDs=('id', lambda x: list(x)),
    flat_block1=('disclosure_1', lambda x: list(x)[0] if len(x) > 0 else ''),
    dataset_block1=('dataset_1', lambda x: list(x)[0] if len(x) > 0 else ''),
).reset_index()
print('Ячейки 2×2 контрбалансировки:')
display(cell_table[['disclosure_order','dataset_order','flat_block1','dataset_block1','N','IDs']])

# Распределение датасетов внутри условий
cond_long = load_conditions()
print('\nДатасет внутри условия flat (из какой ячейки):')
flat_rows = cond_long[cond_long['condition'] == 'flat']
print(flat_rows['dataset'].value_counts().to_dict())
print('\nДатасет внутри условия sections:')
sect_rows = cond_long[cond_long['condition'] == 'sections']
print(sect_rows['dataset'].value_counts().to_dict())

print('\n⚠ Если alpha/beta неравномерно распределены по условиям — '
      'корреляции по срезам condition/dataset носят exploratory-характер.')
print(f'Для сбалансированного анализа используется df_balanced: {balanced_pids}')

In [ ]:
# Проверка эффекта порядка блоков: NASA-TLX block1 vs block2
nasa_wide = nasa_df.pivot(index='id', columns='block', values='nasa_unweighted')
nasa_wide.columns = ['block1_nasa', 'block2_nasa']
nasa_wide = nasa_wide.dropna()

fig, ax = plt.subplots(figsize=(7, 5))
for _, row in nasa_wide.iterrows():
    ax.plot([1, 2], [row['block1_nasa'], row['block2_nasa']],
            '-o', color='gray', alpha=0.6, lw=1, ms=5)
ax.plot([1, 2],
        [nasa_wide['block1_nasa'].mean(), nasa_wide['block2_nasa'].mean()],
        '-o', color='black', lw=2.5, ms=8, label='Среднее')
ax.set_xticks([1, 2])
ax.set_xticklabels(['Блок 1', 'Блок 2'])
ax.set_ylabel('NASA-TLX (невзвешенный)')
ax.set_title('Эффект порядка блоков: NASA-TLX')
ax.legend()
plt.tight_layout()
display(fig)
plt.close(fig)

if len(nasa_wide) >= 4:
    stat, p = stats.wilcoxon(nasa_wide['block1_nasa'], nasa_wide['block2_nasa'])
    print(f'Wilcoxon block1 vs block2: W={stat:.1f}, p={p:.4f}')
    print('→', 'Значимый сдвиг!' if p < 0.05 else 'Систематического сдвига нет (p ≥ 0.05)')
else:
    print('Недостаточно пар для теста Уилкоксона')

### 5b. Эффект порядка блоков — ET-метрики (блок 1 vs блок 2)

Расширение контрбалансировочного анализа на ET-метрики.  
Парный Wilcoxon блок 1 vs блок 2 — если значим, порядок блоков влияет на метрику независимо от условия.


In [ ]:
from pd_analysis import wilcoxon_block_order_et

print('=== Порядок блоков: ET-метрики (блок 1 vs блок 2) ===')
block_order_et = wilcoxon_block_order_et(df_all)
display(block_order_et)
print('\n⚠ Значимые результаты → интерпретировать тесты условия с осторожностью (порядковый эффект).')


---
## Секция 6. Описательная статистика — ET-метрики

### Формальные гипотезы

| # | Гипотеза | Направление | Основная метрика | Тест | Primary df |
|---|---------|-------------|-----------------|------|-----------|
| H1 | Нормализованный диаметр зрачка будет **меньше** в conditions sections, чем flat | sections < flat | `pupil_pct_change_avg` | Wilcoxon (less) | df_good |
| H2 | Число фиксаций на задачу **будет различаться** в зависимости от условия | ненаправленная | `fixation_count` | Wilcoxon (two-sided) | df_good |
| H3 | Суммарный балл NASA-TLX после блока sections будет **ниже** | sections < flat | `nasa_unweighted`, `nasa_weighted` | Wilcoxon (less) | nasa_balanced |
| H4 | Медианное время выполнения задачи будет **короче** в sections | sections < flat | `completion_ms` (медиана) | Wilcoxon (less) | df_all |

**Иерархия датафреймов:**
- Зрачковые метрики (H1): **primary = df_good** (исключены 1818, 2746 с >30% невалидных зрачков); df_balanced и df_all — sensitivity
- Фиксации (H2): **primary = df_good**; df_balanced и df_all — sensitivity
- Время выполнения (H4): **primary = df_all**; df_good — sensitivity
- NASA-TLX (H3): **primary = nasa_balanced** (N=8, сбалансированная 2×2); nasa_df (N=10) — sensitivity

**Когнитивная интерпретация метрик**:

| Метрика | Когнитивный коррелят | Ожидаемое направление |
|---------|---------------------|----------------------|
| pupil_pct_change_avg | Когнитивная нагрузка (% от ITI baseline) | flat > sections (H1) |
| pupil_z_avg | Когнитивная нагрузка (z-score к ITI, sensitivity H1) | flat > sections |
| pupil_mean_avg | Абсолютный диаметр (мм) — только в описательных таблицах | — |
| fixation_count | Объём обработки / навигация | различается (H2, ненаправленная) |
| fixation_duration_mean | Глубина локальной обработки | exploratory |
| scanpath_length | Поисковая эффективность | exploratory |
| nasa_unweighted | Субъективная нагрузка (невзвешенная) | flat > sections (H3) |
| nasa_weighted | Субъективная нагрузка (взвешенная) | flat > sections (H3) |
| completion_ms | Время выполнения задачи (медиана) | flat > sections (H4) |

**⚠ Размер выборки**: N=8–10 пар → мощность ~50% для среднего эффекта. Bootstrap CI (95%, 10k итераций) и permutation test прилагаются.


In [ ]:
_rename = {
    'fixation_duration_mean_mean': 'Длит. фиксации M (с)',
    'fixation_duration_mean_std':  'Длит. фиксации SD',
    'fixation_count_mean':         'Число фиксаций M',
    'fixation_count_std':          'Число фиксаций SD',
    'scanpath_length_mean':        'Длина сканпути M',
    'scanpath_length_std':         'Длина сканпути SD',
    'pupil_pct_change_avg_mean':   'Δ зрачок M (% baseline)',
    'pupil_pct_change_avg_std':    'Δ зрачок SD',
}

cond_all  = summary_by_condition(df_all)
cond_good = summary_by_condition(df_good)

print(f'=== Все участники (N={df_all["participant"].nunique()}) ===')
display(cond_all.rename(columns=_rename).round(4))
print(f'\n=== Хорошие записи (N={df_good["participant"].nunique()}) ===')
display(cond_good.rename(columns=_rename).round(4))


In [ ]:
metrics_labels = [
    ('fixation_duration_mean', 'Длит. фиксации (с)'),
    ('fixation_count',         'Число фиксаций'),
    ('scanpath_length',        'Длина сканпути'),
    ('pupil_pct_change_avg',   'Δ зрачок (% baseline)'),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('ET-метрики: flat vs sections', fontsize=13)

for row_i, (cs, title_sfx) in enumerate([
    (cond_all,  f'Все (N={df_all["participant"].nunique()})'),
    (cond_good, f'Good (N={df_good["participant"].nunique()})'),
]):
    for col_i, (metric, label) in enumerate(metrics_labels):
        ax = axes[row_i][col_i]
        vals = cs.set_index('condition')[f'{metric}_mean']
        errs = cs.set_index('condition')[f'{metric}_std']
        colors = ['#2266cc', '#cc4400']
        ax.bar(vals.index, vals.values, yerr=errs.values,
               color=colors[:len(vals)], capsize=5, width=0.5, alpha=0.85)
        ax.set_title(f'{label}\n({title_sfx})', fontsize=9)
        ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_metrics_by_condition.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)


### 6.1 Парные тесты Уилкоксона: flat vs sections

**Единица анализа**: среднее ET-метрики по участнику внутри условия (агрегация по 6 заданиям).

**H1** (зрачок, направленная): `alternative='greater'` → H_1: median(flat − sections) > 0 → flat > sections → sections < flat.  
Primary df_good; sensitivity — `pupil_z_avg` и df_balanced.  
`pupil_mean_avg` — только в описательной таблице (абсолютный диаметр, не нормирован к ITI).

**H2** (фиксации, ненаправленная): `alternative='two-sided'`. Primary df_good; sensitivity — df_balanced и df_all.

**Bootstrap CI (10k итераций)**: даёт честную картину неопределённости при N=8.  
**Permutation test (10k перестановок)**: exact p без асимптотических допущений.  
**FDR (Benjamini-Hochberg)**: раздельно для каждой группы метрик.


In [ ]:
from pd_analysis import (wilcoxon_condition_test, PUPIL_METRICS, NON_PUPIL_METRICS)

# ── H1: Зрачок (направленная, flat > sections) ───────────────────────────
print('=== H1: Зрачковая нагрузка (primary df_good, alternative=greater → flat>sections) ===')
h1_primary = wilcoxon_condition_test(
    df_good,
    metrics=['pupil_pct_change_avg'],
    alternative='greater',
    n_boot=10000, n_perm=10000,
)
display(h1_primary)

print('\n--- H1 sensitivity: pupil_z_avg ---')
h1_sensitivity = wilcoxon_condition_test(
    df_good,
    metrics=['pupil_z_avg'],
    alternative='greater',
    n_boot=10000, n_perm=10000,
)
display(h1_sensitivity)

print('\n--- H1 sensitivity: df_balanced ---')
h1_balanced = wilcoxon_condition_test(
    df_balanced,
    metrics=PUPIL_METRICS,
    alternative='greater',
    n_boot=10000, n_perm=10000,
)
display(h1_balanced)

# ── H2: Фиксации (ненаправленная) ────────────────────────────────────────
# Primary: df_good (участники с валидным трекингом)
print('\n=== H2: Фиксации (primary df_good, alternative=two-sided) ===')
h2_res = wilcoxon_condition_test(
    df_good,
    metrics=['fixation_count'],
    alternative='two-sided',
    n_boot=10000, n_perm=10000,
)
display(h2_res)

print('\n--- H2 sensitivity: df_balanced ---')
h2_balanced = wilcoxon_condition_test(
    df_balanced,
    metrics=['fixation_count'],
    alternative='two-sided',
    n_boot=10000, n_perm=10000,
)
display(h2_balanced)

print('\n--- H2 sensitivity: df_all ---')
h2_all = wilcoxon_condition_test(
    df_all,
    metrics=['fixation_count'],
    alternative='two-sided',
    n_boot=10000, n_perm=10000,
)
display(h2_all)

# ── Exploratory: прочие ET-метрики ────────────────────────────────────────
print('\n=== Exploratory ET (df_good, two-sided) ===')
wtest_exploratory = wilcoxon_condition_test(
    df_good,
    metrics=NON_PUPIL_METRICS,
    alternative='two-sided',
    n_boot=10000, n_perm=10000,
)
display(wtest_exploratory)

### 6.2 H3: NASA-TLX flat vs sections (парный Wilcoxon)

**H3**: Суммарный балл NASA-TLX после блока sections будет **ниже** (sections < flat).  
Тест: односторонний Wilcoxon (alternative='less') на `nasa_unweighted` и `nasa_weighted`.  
Единица анализа: среднее по блоку на участника × условие (у каждого участника один блок flat и один sections).  
**Primary df**: `nasa_balanced` (N=8, сбалансированная 2×2 выборка).  
`nasa_df` (N=10) — sensitivity check (см. секцию 4b).


In [ ]:
from pd_analysis import wilcoxon_nasa_condition_test

print('=== H3: NASA-TLX flat vs sections (alternative=less, nasa_balanced N=8) ===')
h3_res = wilcoxon_nasa_condition_test(nasa_balanced, n_boot=10000, n_perm=10000)
display(h3_res)

### 6.3 H4: Время выполнения flat vs sections (медианный Wilcoxon)

**H4**: Медианное время выполнения задачи будет **короче** в sections (sections < flat).  
Тест: односторонний Wilcoxon (alternative='less') на медианном `completion_ms` по участнику × условие.  
Primary df_all (не-зрачковая метрика).


In [ ]:
from pd_analysis import wilcoxon_completion_test

print('=== H4: Медианное completion_ms flat vs sections (alternative=less) ===')
h4_res = wilcoxon_completion_test(df_all, n_boot=10000, n_perm=10000)
display(h4_res)

print('\n--- H4 sensitivity: df_good ---')
h4_good = wilcoxon_completion_test(df_good, n_boot=10000, n_perm=10000)
display(h4_good)


### 6.4 Сводная таблица результатов по гипотезам

Центральный артефакт раздела результатов: все H1–H4 в одной таблице.  
**Колонки**: hypothesis → metric → N → test → W → p_raw → p_perm → p_FDR → r → d → CI_r → CI_d → decision


In [ ]:
from pd_analysis import build_hypothesis_summary

# Собираем сводную таблицу H1-H4 (full sample)
hypothesis_summary = build_hypothesis_summary(
    h1_primary=h1_primary,
    h1_sensitivity=h1_sensitivity,
    h2_res=h2_res,
    h3_res=h3_res,
    h4_res=h4_res,
    subsample_label='full',
)
print('=== СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ H1-H4 ===')
display(hypothesis_summary)


### 6.x GEE + LMM: condition + dataset + block_num + (1|participant)

**GEE (primary)**: Generalised Estimating Equations с exchangeable working correlation — устойчивее LMM при малом числе кластеров (N=8 участников). Оценивает population-averaged effect.

**LMM (exploratory)**: REML=True, оптимизатор lbfgs. При N=8 уровнях случайного эффекта возможен singular fit (дисперсия RE ≈ 0). Колонка `converged=False` помечает нестабильные подгонки. Рассматривать только как дополнение к GEE.

Обе модели контролируют датасет (alpha/beta) и порядок блоков.


In [ ]:
import importlib
if importlib.util.find_spec('statsmodels') is None:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])

from pd_analysis import gee_condition_test, lmm_condition_test, PUPIL_METRICS, NON_PUPIL_METRICS

# ── GEE (primary) ─────────────────────────────────────────────────────────
print('=== GEE (primary, exchangeable WC): условие flat vs sections ===')
gee_pupil  = gee_condition_test(df_good,  metrics=PUPIL_METRICS)
gee_nonpup = gee_condition_test(df_all,   metrics=NON_PUPIL_METRICS)

print('--- GEE: Зрачок (df_good) ---')
display(gee_pupil)
print('\n--- GEE: Не-зрачок (df_all) ---')
display(gee_nonpup)

# ── LMM (exploratory) ─────────────────────────────────────────────────────
print('\n=== LMM (exploratory, REML): ⚠ singular fit вероятен при N=8 ===')
lmm_pupil  = lmm_condition_test(df_good,  metrics=PUPIL_METRICS)
lmm_nonpup = lmm_condition_test(df_all,   metrics=NON_PUPIL_METRICS)

print('--- LMM: Зрачок (df_good) ---')
display(lmm_pupil)
print('\n--- LMM: Не-зрачок (df_all) ---')
display(lmm_nonpup)

not_converged = lmm_pupil[lmm_pupil.get('converged', True) == False]
if len(not_converged) > 0:
    print(f'\n⚠ Не сошлось: {not_converged["metric"].tolist()} — результаты exploratory.')


In [ ]:
task_all  = summary_by_task(df_all)
task_good = summary_by_task(df_good)

print(f'=== Все участники ===')
display(task_all.rename(columns=_rename).round(4))
print(f'\n=== Хорошие записи (N={df_good["participant"].nunique()}) ===')
display(task_good.rename(columns=_rename).round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'ET-метрики по заданию (хорошие записи, N={df_good["participant"].nunique()}, M ± SD)', fontsize=13)
axes_flat = axes.flatten()
x = range(len(TASK_ORDER))
ts = task_good.set_index('task_id')

for ax, (metric, label) in zip(axes_flat, metrics_labels):
    avail = [t for t in TASK_ORDER if t in ts.index]
    means = ts.loc[avail, f'{metric}_mean'].values.astype(float)
    stds  = ts.loc[avail, f'{metric}_std'].values.astype(float)
    colors = ['#2266cc'] * 6 + ['#cc4400'] * 6
    ax.bar(range(len(avail)), means, yerr=stds,
           color=colors[:len(avail)], capsize=4, alpha=0.8)
    ax.set_xticks(range(len(avail)))
    ax.set_xticklabels(avail, fontsize=8)
    ax.set_title(label, fontsize=10)
    ax.axvline(5.5, color='gray', lw=0.8, ls='--')

legend_el = [
    mpatches.Patch(color='#2266cc', label='Alpha (A1–A6)'),
    mpatches.Patch(color='#cc4400', label='Beta (B1–B6)'),
]
fig.legend(handles=legend_el, loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_metrics_by_task.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)


---
## Секция 7. Валидность измерения когнитивной нагрузки (ET × NASA-TLX)

**Как читать таблицы корреляций:**
- Строки = айтрекинговые метрики (число фиксаций, длительность фиксации, длина сканпути, диаметр зрачка)
- Столбцы = шкалы NASA-TLX (невзвешенный/взвешенный итог + 5 субшкал)
- Числа = коэффициент корреляции Спирмена r (от −1 до +1): чем ближе к ±1, тем сильнее связь
- **Жёлтая подсветка** = p < 0.05, статистически значимая корреляция
- \* p < 0.05; \*\* p < 0.01

**Методологическая заметка о разрезах:**  
Корреляции считаются на уровне **участник × блок** (N ≤ 20 для df_all). Это корректная единица анализа,
поскольку NASA-TLX заполняется один раз в конце каждого блока — после 6 заданий подряд.  
Разрезы по типу задачи (Lookup/Comparison/Diagnosis) для корреляций **не делаются**: NASA-TLX
отражает нагрузку за весь блок, а не за отдельный тип; кроме того, N на ячейку был бы ≤ 7.  
Разрезы по условию (flat/sections) и датасету (alpha/beta) показывают, устойчива ли связь внутри
каждого условия — это exploratory-анализ с малым N, интерпретировать осторожно.

### 7a. Общие корреляции Спирмена (основной результат)

In [ ]:
# nasa_df для good и balanced подвыборок (фильтруем по id)
nasa_good     = nasa_df[nasa_df['id'].isin(good_pids)]
nasa_balanced = nasa_df[nasa_df['id'].isin(balanced_pids)]

print(f'=== Все участники (N={df_all["participant"].nunique()}, {len(df_all)//12*2} obs.) ===')
corr_r_all, corr_p_all, _ = spearman_nasa_eyetrack(df_all, nasa_df)
display(corr_r_all.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}'))
display(corr_p_all.style.highlight_between(left=0, right=0.05, color='#ffe08a').format('{:.4f}'))

print(f'\n=== Хорошие записи (N={df_good["participant"].nunique()}) ===')
corr_r_good, corr_p_good, _ = spearman_nasa_eyetrack(df_good, nasa_good)
display(corr_r_good.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}'))
display(corr_p_good.style.highlight_between(left=0, right=0.05, color='#ffe08a').format('{:.4f}'))

from pd_analysis import apply_fdr_to_spearman

corr_p_good_fdr = apply_fdr_to_spearman(corr_p_good)
n_tests = corr_p_good.shape[0] * corr_p_good.shape[1]
print(f'\n=== FDR-поправка Бенджамини–Хохберга ({n_tests} тестов) ===')
display(corr_p_good_fdr.style
    .highlight_between(left=0, right=0.05, color='#ffe08a').format('{:.4f}'))


In [ ]:
heatmap_path = os.path.join(DATA_ROOT, 'pd_spearman_heatmap.png')
fig = plot_spearman_heatmap(corr_r_good, corr_p_good, save_path=heatmap_path, show=False)
display(fig)
plt.close(fig)

### 7b. По условию (flat / sections) — exploratory

**Иерархия датафреймов** (исправлено):
- `df_good` (primary) — исключены участники с плохим зрачком (1818, 2746); используется для зрачковых ET-метрик
- `df_all` (sensitivity) — все участники
- `df_balanced` (sensitivity) — 2 per ячейка 2×2, но включает 1818 и 2746, поэтому не основной для зрачка

⚠ Стратификация по условию: N≈4–5 наблюдений на группу → исключительно exploratory, CI ≈±0.8.


In [ ]:
for label, gdf, ndf in [
    ('Good (primary)', df_good, nasa_good),
    ('All (sensitivity)', df_all, nasa_df),
    ('Balanced (sensitivity)', df_balanced, nasa_balanced),
]:
    res = spearman_by_split(gdf, ndf, split_col='condition')
    for split_val, (r_df, p_df, _) in res.items():
        p_fdr = apply_fdr_to_spearman(p_df)
        print(f'\n=== {label} | condition={split_val} (N={len(gdf[gdf["condition"]==split_val]["participant"].unique())}) ===')
        display(r_df.round(2))
        print('FDR-p:')
        display(p_fdr.round(3))


### 7c. По датасету (alpha / beta) — exploratory

Аналогично: `df_balanced` как основной + `df_all` как sensitivity check.

**Зачем разбивка по датасету**: alpha (A1–A6) и beta (B1–B6) — разные наборы задач.  
Если ET-NASA-TLX корреляции существенно различаются между датасетами, это указывает на разную когнитивную сложность или структуру задач, а не только на эффект условия.  
С учётом дизайна (половина участников начинала с alpha, другая — с beta), разбивка по датасету также частично отражает эффект порядка блоков — разграничить их при N=4 невозможно.


In [ ]:
for label, gdf, ndf in [
    ('Balanced (основной)', df_balanced, nasa_balanced),
    ('All (sensitivity check)', df_all, nasa_df),
]:
    res = spearman_by_split(gdf, ndf, split_col='dataset')
    print(f'\n=== {label} ===')
    for ds_val, (r_df, p_df, sub) in res.items():
        print(f'--- Датасет: {ds_val.upper()}  (N={len(sub)} obs.) ---')
        display(r_df.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}'))
        display(p_df.style.highlight_between(left=0, right=0.05, color='#ffe08a').format('{:.4f}'))

---
## Секция 8. Детальный анализ ET — Зрачок (основная метрика)

Зрачок — основная метрика когнитивной нагрузки для данного эксперимента, т.к. точность позиции взгляда могла варьироваться.

### 8.1 Дельта зрачка от ITI-baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Изменение зрачка от ITI-baseline по заданиям (M ± SD)', fontsize=12)

for ax, (df_sub, title) in zip(axes, [
    (df_all,  f'Все участники (N={df_all["participant"].nunique()})'),
    (df_good, f'Хорошие записи (N={df_good["participant"].nunique()})'),
]):
    pupil_task = (df_sub.groupby('task_id')[['pupil_pct_change_avg']]
                  .agg(['mean', 'std'])
                  .round(3))
    pupil_task.columns = ['mean', 'std']
    avail  = [t for t in TASK_ORDER if t in pupil_task.index]
    means  = pupil_task.loc[avail, 'mean'].values
    stds   = pupil_task.loc[avail, 'std'].values
    colors = ['#2266cc'] * 6 + ['#cc4400'] * 6

    ax.bar(range(len(avail)), means, yerr=stds,
           color=colors[:len(avail)], capsize=4, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.axvline(5.5, color='gray', lw=0.8, ls=':')
    ax.set_xticks(range(len(avail)))
    ax.set_xticklabels(avail, fontsize=8)
    ax.set_ylabel('Δ зрачок от baseline (%)')
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_pupil_delta.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)

# Числовые данные
pupil_tab_good = (df_good.groupby(['task_id','condition'])[['pupil_pct_change_avg']]
                  .agg(['mean','std']).round(3))
display(pupil_tab_good)

### 8.2 Трассы зрачка по условию

flat = синий (#2266cc), sections = оранжевый (#cc4400).  
Оттенки — разные участники внутри одного условия.  
**Rug у основания** (10% непрозрачности) = невалидные сэмплы зрачка (LPV=0 AND RPV=0) — видно, где данных нет и сколько их.

> Графики могут отрисовываться 1–2 минуты из-за загрузки TSV-файлов.

In [ ]:
print('Все участники — трассы по условию (может занять ~1 мин):')
fig_c = plot_pupil_traces(
    df_all, split_by='condition',
    use_cleaned=True,
    save_path=os.path.join(DATA_ROOT, 'pd_pupil_traces_condition_all.png'),
    show=False,
)
display(fig_c)
plt.close(fig_c)

In [ ]:
print('Хорошие записи — трассы по условию:')
fig_cg = plot_pupil_traces(
    df_good, split_by='condition',
    participant_ids=good_pids,
    use_cleaned=True,
    save_path=os.path.join(DATA_ROOT, 'pd_pupil_traces_condition_good.png'),
    show=False,
)
display(fig_cg)
plt.close(fig_cg)

### 8.3 Трассы зрачка по датасету

alpha = жёлто-зелёный (#55aa44), beta = фиолетовый (#9944cc); полупрозрачность = разные участники.

**Контекст**: alpha (A1–A6) и beta (B1–B6) — разные наборы задач.  
Для половины участников alpha — первый блок (новизна, более высокая нагрузка ожидается);  
для другой половины alpha — второй блок (знакомость с форматом).  
Систематические различия в базовом уровне зрачка между alpha и beta могут указывать на эффект порядка или на объективно разную сложность задач.  
При анализе следует учитывать, что разбивка по датасету и разбивка по порядку блоков конфаундированы в данном дизайне.


In [ ]:
fig_d = plot_pupil_traces(
    df_all, split_by='dataset',
    use_cleaned=True,
    save_path=os.path.join(DATA_ROOT, 'pd_pupil_traces_dataset.png'),
    show=False,
)
display(fig_d)
plt.close(fig_d)

### 8.4 Сводный нормализованный график зрачка по заданию

Ось X — нормированное время выполнения задачи [0, 1]; ось Y — диаметр зрачка (мм, среднее по участникам).

**Зачем временна́я нормализация**: задачи разного типа имеют разную длительность (Lookup быстрее Diagnosis).  
Нормировка позволяет сравнивать внутризадачную динамику зрачка независимо от абсолютного времени,  
и выявлять, в какой фазе задачи (начало / середина / конец) нагрузка максимальна.  
Ожидаемый паттерн (cognitive load hypothesis): рост зрачка в первые 30–50% задачи (обработка условия),  
затем плато или спад (выполнение автоматизированных действий).


In [ ]:
summary_plot = os.path.join(DATA_ROOT, 'pd_pupil_by_task.png')
fig = plot_pupil_by_task(df_good, save_path=summary_plot, show=False)
display(fig)
plt.close(fig)

### 8.5 Групповое усреднение: flat vs sections и alpha vs beta

Каждая трасса = один участник × одно задание, нормированное на [0, 1].  
Линия = среднее по группе; полоса = ±1 SEM (стандартная ошибка среднего).  
Видимый дисбаланс в легенде (разное число участников на условие) — следствие design unbalance.

**Как интерпретировать**: если ±SEM-полосы flat и sections не перекрываются на значительной части трассы,  
это визуально подтверждает эффект условия на зрачковый ответ — согласуется с Wilcoxon-результатом для `fixation_count`.  
Перекрытие полос на большей части трассы говорит о малом или нестабильном эффекте.  
Ранняя дивергенция (первые 20–30% задачи) указывала бы на различия в первоначальной ориентации в интерфейсе,  
поздняя — на различия в фазе ответа.


In [ ]:
# flat vs sections
fig_gc = plot_pupil_by_group(
    df_good, split_by='condition',
    save_path=os.path.join(DATA_ROOT, 'pd_pupil_group_condition.png'),
    show=False,
)
display(fig_gc)
plt.close(fig_gc)

In [ ]:
# alpha vs beta
fig_gd = plot_pupil_by_group(
    df_good, split_by='dataset',
    save_path=os.path.join(DATA_ROOT, 'pd_pupil_group_dataset.png'),
    show=False,
)
display(fig_gd)
plt.close(fig_gd)

---
## Секция 9. Детальный анализ ET — Фиксации и сканпуть

**Фиксации и сканпуть как дополнительные индикаторы когнитивной нагрузки**:  
в отличие от зрачкового ответа (автономный, непроизвольный), фиксации отражают сознательное распределение зрительного внимания.

| Метрика | Что отражает |
|---------|-------------|
| `fixation_count` | Общий объём зрительной обработки; больше = больше «точек» интереса |
| `fixation_duration_mean` | Глубина локальной обработки; длиннее = больше усилий на понимание |
| `scanpath_length` | Эффективность поиска; длиннее = менее эффективная навигация |

**Гипотезы для секций vs flat**:  
- Sections-формат структурирует контент → участники делают **больше, но более коротких** фиксаций (navigational pattern);  
- Flat-формат требует самостоятельного структурирования → **длиннее** средняя фиксация и **длиннее** сканпуть.  
Согласованность с паттерном зрачка (секция 8) будет говорить о мультиметрической валидности эффекта.


In [ ]:
fix_metrics = [
    ('fixation_count',         'Число фиксаций'),
    ('fixation_duration_mean', 'Длит. фиксации (с)'),
    ('scanpath_length',        'Длина сканпути'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Метрики фиксаций: flat vs sections (верхний ряд — все, нижний — good)', fontsize=12)

for row_i, (df_sub, n_lbl) in enumerate([
    (df_all,  f'Все (N={df_all["participant"].nunique()})'),
    (df_good, f'Good (N={df_good["participant"].nunique()})'),
]):
    agg = df_sub.groupby(['participant','condition'])[
        [m for m, _ in fix_metrics]
    ].mean().reset_index()

    for col_i, (metric, label) in enumerate(fix_metrics):
        ax = axes[row_i][col_i]
        flat_v = agg[agg['condition'] == 'flat'][metric].values
        sect_v = agg[agg['condition'] == 'sections'][metric].values

        parts = ax.violinplot([flat_v, sect_v], positions=[1, 2],
                              showmedians=True, showextrema=False)
        for pc, color in zip(parts['bodies'], ['#2266cc', '#cc4400']):
            pc.set_facecolor(color)
            pc.set_alpha(0.4)
        parts['cmedians'].set_color('black')

        pf = agg[agg['condition'] == 'flat'].set_index('participant')[metric]
        ps = agg[agg['condition'] == 'sections'].set_index('participant')[metric]
        for pid in pf.index.intersection(ps.index):
            ax.plot([1, 2], [pf[pid], ps[pid]], '-', color='gray', alpha=0.5, lw=0.8)

        ax.scatter([1] * len(flat_v), flat_v, c='#2266cc', s=30, zorder=5, alpha=0.8)
        ax.scatter([2] * len(sect_v), sect_v, c='#cc4400', s=30, zorder=5, alpha=0.8)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(['Flat', 'Sections'])
        ax.set_title(f'{label}\n({n_lbl})', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_fixation_metrics.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)


### 9.2 По типу задачи × условие (хорошие записи)

**Ожидаемый градиент когнитивной нагрузки**: Lookup < Comparison < Diagnosis.  
Теория когнитивной нагрузки предсказывает: диагностические задачи требуют интеграции информации из нескольких источников → больше фиксаций, длиннее сканпуть, выше зрачковый ответ.

**Как интерпретировать графики**:  
- Если градиент прослеживается — ET-метрики чувствительны к сложности задачи, что повышает их конструктную валидность.  
- Если градиент отсутствует — возможно, сложность задач не варьировалась достаточно, или N слишком мал для детекции.  
- Взаимодействие тип×условие: если эффект flat vs sections сильнее для Diagnosis, чем для Lookup,  
  sections-формат может особенно помогать при высококогнитивных задачах — это имеет прямое прикладное значение для дизайна интерфейсов.


In [ ]:
all_metrics_plot = fix_metrics + [('pupil_pct_change_avg', 'Δ зрачок (% baseline)')]
task_types_order = ['Lookup', 'Comparison', 'Diagnosis']

fig, axes = plt.subplots(len(all_metrics_plot), len(task_types_order),
                          figsize=(14, 4 * len(all_metrics_plot)))
fig.suptitle('ET-метрики: условие × тип задачи (хорошие записи)', fontsize=12)

for row_i, (metric, label) in enumerate(all_metrics_plot):
    for col_i, tt in enumerate(task_types_order):
        ax = axes[row_i][col_i]
        sub = df_good[df_good['task_type'].str.lower() == tt.lower()]
        flat_v = sub[sub['condition'] == 'flat'][metric].dropna().values
        sect_v = sub[sub['condition'] == 'sections'][metric].dropna().values

        if len(flat_v) == 0 and len(sect_v) == 0:
            ax.set_visible(False)
            continue

        bp = ax.boxplot([flat_v, sect_v], positions=[1, 2],
                        patch_artist=True, widths=0.5, showfliers=False)
        for box, color in zip(bp['boxes'], ['#2266cc', '#cc4400']):
            box.set_facecolor(color)
            box.set_alpha(0.6)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(['Flat', 'Sections'], fontsize=8)
        ax.set_title(f'{label}\n{tt}', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_ROOT, 'pd_metrics_by_tasktype.png'), dpi=150, bbox_inches='tight')
display(fig)
plt.close(fig)


### 9.3 Формальный тест взаимодействия condition × task_type

LMM с interaction term: `metric ~ C(condition) * C(task_type) + block_num + (1|participant)`.  
Особый интерес: `C(condition)[T.sections]:C(task_type)[T.Diagnosis]` — помогает ли progressive disclosure именно при высоконагрузочных задачах (Diagnosis)?


In [ ]:
from pd_analysis import lmm_interaction_test

print('=== Взаимодействие condition × task_type ===')
interaction_res = lmm_interaction_test(
    df_all,
    metrics=['fixation_count', 'pupil_pct_change_avg', 'completion_ms']
)
if not interaction_res.empty:
    display(interaction_res)
else:
    print('Нет результатов (task_type может отсутствовать в df_all).')


### 9.4 Sensitivity check: подвыборка easy_tasks (A1–A4, B1–B4)

Задачи A5, A6, B5, B6 слишком различаются между датасетами alpha и beta и могут провоцировать межгрупповые различия, не связанные с условием. Исключение задач 5–6 (`easy_tasks`) даёт более чистое сравнение flat vs sections.


In [ ]:
from pd_analysis import filter_easy_tasks, build_hypothesis_summary

df_easy      = filter_easy_tasks(df_all)
df_easy_good = filter_easy_tasks(df_good)

print(f'easy_tasks: {len(df_easy)} строк (было {len(df_all)}), участников: {df_easy["participant"].nunique()}')

# H1 (зрачок) на easy_tasks
h1_easy = wilcoxon_condition_test(
    df_easy_good, metrics=['pupil_pct_change_avg'],
    alternative='greater', n_boot=10000, n_perm=10000
)
# H2 на easy_tasks (primary df_easy_good)
h2_easy = wilcoxon_condition_test(
    df_easy_good, metrics=['fixation_count'],
    alternative='two-sided', n_boot=10000, n_perm=10000
)
# H4 на easy_tasks
h4_easy = wilcoxon_completion_test(df_easy, n_boot=10000, n_perm=10000)

summary_easy = build_hypothesis_summary(
    h1_primary=h1_easy,
    h2_res=h2_easy,
    h4_res=h4_easy,
    subsample_label='easy_tasks',
)
print('=== Сводная таблица (easy_tasks) ===')
display(summary_easy)


---
## Секция 10. Сохранение сводного Excel

Результаты всех анализов сохраняются в `data/pd_group_summary.xlsx` (15 листов):

| Лист | Содержимое |
|------|------------|
| `per_task_*` | Данные на уровне задания по всем подвыборкам |
| `wilcoxon_*` | Таблицы Wilcoxon-тестов с FDR-поправкой и эффект-сайзами |
| `spearman_r_*`, `spearman_p_*` | Матрицы корреляций Спирмена (r и p) |
| `nasa_summary` | Сводка NASA-TLX по условию и датасету |
| `pupil_metrics` | Агрегированные метрики зрачка по участнику |

Файл предназначен для отчётности и дополнительного анализа в Excel/R.  
При изменении данных или параметров — перезапустить ноутбук с начала и выполнить эту ячейку повторно.


In [ ]:
from pd_analysis import OUT_XLSX

corr_r_balanced, corr_p_balanced, _ = spearman_nasa_eyetrack(df_balanced, nasa_balanced)

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    df_all.to_excel(writer,      sheet_name='per_task_all',      index=False)
    df_good.to_excel(writer,     sheet_name='per_task_good',     index=False)
    df_balanced.to_excel(writer, sheet_name='per_task_balanced', index=False)
    quality_df.to_excel(writer,  sheet_name='data_quality',      index=False)
    summary_by_condition(df_all).to_excel(writer,  sheet_name='by_condition_all',  index=False)
    summary_by_condition(df_good).to_excel(writer, sheet_name='by_condition_good', index=False)
    summary_by_task(df_all).to_excel(writer,  sheet_name='by_task_all',  index=False)
    summary_by_task(df_good).to_excel(writer, sheet_name='by_task_good', index=False)
    stai_df.to_excel(writer,     sheet_name='stai_scores',       index=False)
    nasa_df.to_excel(writer,     sheet_name='nasa_tlx',          index=False)
    corr_r_all.to_excel(writer,       sheet_name='spearman_r_all')
    corr_r_good.to_excel(writer,      sheet_name='spearman_r_good')
    corr_r_balanced.to_excel(writer,  sheet_name='spearman_r_balanced')
    corr_p_good.to_excel(writer,      sheet_name='spearman_p_good')
    corr_p_balanced.to_excel(writer,  sheet_name='spearman_p_balanced')

print(f'Сохранено: {OUT_XLSX}')
print(f'Листов: {len(pd.ExcelFile(OUT_XLSX).sheet_names)}')